# Điều chỉnh siêu tham số có giới hạn

Bước 11 của kế hoạch S6E11. [Kế hoạch dự án](../../../plan.md#L298)

T01-M50 đổi hệ số làm trơn từ 20 lên 50. GLM thử C và bộ giải Newton riêng. Cây thử điều chuẩn, tốc độ học và lấy mẫu. File metric lưu log-loss, số vòng cùng trạng thái hội tụ khi trình chạy thu được các trường đó.

## Chạy lại

Chạy từ thư mục `s6e11_gpu/plan_runs` với môi trường Python của dự án và GPU đã kiểm tra. Đặt `S6E11_DATA_DIR` hoặc thêm `--data-dir` nếu CSV chính thức ở thư mục khác. Trình chạy tiếp tục từ checkpoint có cấu hình khớp. Cell dưới đọc kết quả theo mặc định.

```sh
python -B -X utf8 run_plan.py --stage advanced --compare-xgb-digits --tune-family xgb --seed-anchor XGB-D03
```

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

here = Path.cwd().resolve()
roots = [Path(os.environ["S6E11_PLAN_RUNS"])] if "S6E11_PLAN_RUNS" in os.environ else []
for base in (here, *here.parents):
    roots.extend((base, base / "plan_runs", base / "INT3405E_FD" / "s6e11_gpu" / "plan_runs",
                  base / "s6e11_gpu" / "plan_runs",
                  base / "Predicting Credit Card Fraud" / "INT3405E_FD" / "s6e11_gpu" / "plan_runs"))
plan_dir = next((path for path in roots if (path / "run_plan.py").is_file()
                 and (path / "report_results.py").is_file()), None)
if plan_dir is None:
    raise FileNotFoundError("Mở notebook trong dự án hoặc đặt S6E11_PLAN_RUNS.")

CHAY_LAI = False
if CHAY_LAI:
    subprocess.run([sys.executable, "-B", "-X", "utf8", str(plan_dir / "run_plan.py"),
                    *['--stage', 'advanced', '--compare-xgb-digits', '--tune-family', 'xgb', '--seed-anchor', 'XGB-D03']],
                   cwd=plan_dir, check=True)

## Kết quả tại thời điểm tạo notebook

Nguồn số liệu là bản tổng hợp công bố từ lượt chạy GPU lịch sử trong `published/full`. Chế độ đọc bản công bố đối chiếu các bảng tổng hợp và mã kiểm tra đã ghi. Tính lại chỉ số từ từng dự đoán cần dữ liệu và kết quả đầy đủ ở máy chạy. [Bảng tổng hợp](../published/full/comparison.csv)

| Mã | Đối chứng | ROC AUC OOF | Δ AUC | AP | Log-loss | Giây | Fold 1 | Fold 2 | Fold 3 | Fold 4 | Fold 5 |
| :--- | :--- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| `TUNE-NOSUB` | `XGB-D03` | 0.950104 | -0.000079 | 0.528598 | 0.044444 | 401.8 | 0.949684 | 0.950965 | 0.954947 | 0.950777 | 0.944443 |
| `TUNE-ETA005` | `XGB-D03` | 0.950060 | -0.000122 | 0.528456 | 0.044458 | 279.8 | 0.949692 | 0.950717 | 0.955062 | 0.950767 | 0.944679 |
| `TUNE-LAMBDA3` | `XGB-D03` | 0.950045 | -0.000137 | 0.528513 | 0.044450 | 360.8 | 0.949733 | 0.950859 | 0.954945 | 0.950742 | 0.944433 |
| `GLM-NEWTON` | `P1-P2` | 0.947656 | +0.000075 | 0.515512 | 0.045275 | 44.2 | 0.946464 | 0.949029 | 0.953155 | 0.947935 | 0.941843 |
| `GLM-C0.3` | `P1-P2` | 0.947605 | +0.000024 | 0.515563 | 0.045281 | 42.5 | 0.946500 | 0.948953 | 0.953050 | 0.947871 | 0.941740 |
| `GLM-C0.1` | `P1-P2` | 0.947558 | -0.000023 | 0.515294 | 0.045300 | 50.9 | 0.946486 | 0.948825 | 0.953019 | 0.947905 | 0.941690 |
| `T01-M50` | `T01` | 0.947208 | -0.000053 | 0.513007 | 0.045436 | 20.9 | 0.946282 | 0.948593 | 0.952604 | 0.947335 | 0.941375 |

## Đọc dữ liệu kết quả hiện tại

Cell này ưu tiên dự đoán OOF và tệp metric cục bộ khi có lượt chạy. Bản sao mới đọc bảng tổng hợp trong `published/full`.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

sys.path.insert(0, str(plan_dir))
from report_results import ARCHIVE_PROVENANCE, final_summary, read_stage, result_source

result_dir, archived = result_source()
if archived:
    display(Markdown(ARCHIVE_PROVENANCE))
else:
    display(Markdown("Nguồn cục bộ. Chỉ số của lượt hoàn tất được tính lại từ dự đoán OOF."))
rows, pending, notes = read_stage('04-tuning', result_dir, archived=archived)
if rows:
    points = [{"mã": row["name"], "đối chứng": row["config"].get("parent"),
               "ROC AUC OOF": row["oof_auc"], "AP": row["average_precision"],
               "delta AUC": row.get("delta_oof_auc"), "log-loss": row["log_loss"],
               **{f"fold {item['fold']}": item["auc"] for item in row["folds"]}}
              for row in rows]
    display(pd.DataFrame(points))
if pending:
    display(Markdown(pending))
if notes:
    display(Markdown("\n".join(f"- {note}" for note in notes)))
if False:
    display(Markdown(final_summary(result_dir, markdown_links=False, archived=archived)))

## Xem cấu hình và log theo fold

Hiển thị lượt có AUC cao nhất trong nhóm hiện tại.

In [ ]:
if rows:
    record = rows[0]
    display(record["config"])
    display(pd.DataFrame([
        {key: item.get(key) for key in
         ("fold", "auc", "device", "seconds", "iterations", "converged", "rounds")}
        for item in record["folds"]
    ]))
    if not archived:
        with np.load(result_dir / record["name"] / "predictions.npz", allow_pickle=False) as saved:
            display({name: saved[name].shape for name in ("ids", "fold", "oof", "test_ids", "test")})

## Phạm vi diễn giải

Các cấu hình được so trên bộ fold khảo sát. Bảng chỉ mô tả các cấu hình đã chạy.